# 11 · Errors & Context Managers

Real data is dirty and systems fail. Robust pipelines *expect* errors and handle
them deliberately. This notebook covers exceptions and the `with` statement that
guarantees cleanup.

## Exceptions and `try` / `except`

When something goes wrong Python **raises** an exception. Unhandled, it stops the
program. `try/except` lets you catch specific error types and respond. Catch the
*narrowest* exception you can — never a bare `except:`.

In [ ]:
def to_int(value):
    try:
        return int(value)
    except ValueError:
        return None          # couldn't parse -> treat as missing

for v in ['42', '3.5', '', 'abc', '  7 ']:
    print(repr(v), '->', to_int(v))

## `else` and `finally`

`else` runs only if no exception was raised; `finally` runs **no matter what**
— the place for cleanup (closing files, connections).

In [ ]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('cannot divide by zero')
        return None
    else:
        print('division ok')
        return result
    finally:
        print('...cleanup always runs')

print(divide(10, 2))
print(divide(10, 0))

## Common exception types

Knowing the built-in exceptions helps you catch precisely and read tracebacks.

In [ ]:
examples = [
    ('int("x")', lambda: int('x')),          # ValueError
    ('[1,2][5]', lambda: [1, 2][5]),            # IndexError
    ('{}["k"]', lambda: {}['k']),             # KeyError
    ('1 + "a"', lambda: 1 + 'a'),             # TypeError
    ('open("nope")', lambda: open('nope')),   # FileNotFoundError
]
for label, fn in examples:
    try:
        fn()
    except Exception as e:
        print(f'{label:>14} -> {type(e).__name__}: {e}')

## Raising and custom exceptions

Raise your own errors to signal problems clearly. A custom exception class makes
pipeline failures self-documenting and easy to catch selectively.

In [ ]:
class DataQualityError(Exception):
    '''Raised when a record fails validation.'''

def validate(order):
    if order['amount'] < 0:
        raise DataQualityError(f"negative amount in order {order['id']}")
    return order

try:
    validate({'id': 5, 'amount': -10})
except DataQualityError as e:
    print('rejected:', e)

## Context managers: the `with` statement

A **context manager** guarantees setup and teardown around a block, even if an
error occurs. Opening files is the classic case — `with` closes the file
automatically, so you never leak handles.

In [ ]:
from pathlib import Path
import tempfile

tmp = Path(tempfile.gettempdir()) / 'demo.txt'
with open(tmp, 'w', encoding='utf-8') as f:
    f.write('line 1\nline 2\n')
# file is guaranteed closed here, even if write() had raised

with open(tmp, encoding='utf-8') as f:
    print(f.read().strip())

## Writing your own context manager

`contextlib.contextmanager` turns a generator into a context manager — the code
before `yield` is setup, after `yield` is teardown. Great for timing, temporary
state, or transactions.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f'{label} took {elapsed*1000:.1f} ms')

with timer('sum 1M'):
    total = sum(range(1_000_000))
print('result:', total)

### Recap

`try/except` catches specific exceptions; `else`/`finally` structure success and
cleanup; raise custom exceptions for clear pipeline failures; `with` guarantees
teardown; `@contextmanager` builds your own. That completes **Track 1** — you're
fluent in the language. Next: Track 2 opens with files and `pathlib`.